# Two-Model + Time-Based K-Fold

**Base:** Two-model approach (0.2633)
**Change:** Instead of single train/val split, use 4 time-based folds.

Each fold trains on more data, and test predictions are averaged across all folds.
This creates a more diverse and robust ensemble.

- H1, H25: Combined K-Fold
- H3, H10: Two-model K-Fold (precision + range per fold)

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import warnings
import os
warnings.filterwarnings('ignore')

print('Libraries imported')

Libraries imported


In [2]:
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.parquet'
    TEST_PATH = 'test.parquet'

FORECAST_WINDOWS = [1, 3, 10, 25]
TWO_MODEL_HORIZONS = {3, 10}

# Time-based K-Fold splits
# Each fold: train up to cutoff, validate on the next segment
# Final model for test predictions uses ALL folds
FOLDS = [
    {'train_end': 2800, 'val_start': 2801, 'val_end': 3100},
    {'train_end': 3100, 'val_start': 3101, 'val_end': 3350},
    {'train_end': 3350, 'val_start': 3351, 'val_end': 3601},
]

# For final test predictions, train on everything
FULL_TRAIN_END = 3601

RANGE_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.015, 'n_estimators': 4200,
    'num_leaves': 80, 'min_child_samples': 200,
    'feature_fraction': 0.6, 'bagging_fraction': 0.7,
    'bagging_freq': 5, 'lambda_l1': 0.1, 'lambda_l2': 10.0,
    'verbosity': -1
}

PRECISION_PARAMS = {
    'objective': 'huber', 'metric': 'rmse',
    'learning_rate': 0.01, 'n_estimators': 5000,
    'num_leaves': 100, 'min_child_samples': 80,
    'feature_fraction': 0.7, 'bagging_fraction': 0.75,
    'bagging_freq': 5, 'lambda_l1': 0.01, 'lambda_l2': 2.0,
    'huber_delta': 0.01, 'verbosity': -1
}

SEEDS = [42, 2024, 12345]

print(f'Config: {len(FOLDS)} folds x {len(SEEDS)} seeds = {len(FOLDS) * len(SEEDS)} models per horizon')

Config: 3 folds x 3 seeds = 9 models per horizon


In [3]:
def add_lag_features(df, value_cols=['feature_al', 'feature_am', 'feature_cg', 'feature_by'], lags=[1, 3, 5, 10, 25]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].shift(lag)
    return df

def add_rolling_features(df, value_cols=['feature_al', 'feature_am'], windows=[5, 10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for window in windows:
            df[f'{col}_roll_mean_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
            df[f'{col}_roll_std_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).std())
    return df

def add_trend_features(df, value_cols=['feature_al', 'feature_am'], windows=[10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    def rolling_slope(series, window):
        def calc_slope(y):
            if len(y) < 2:
                return 0
            x = np.arange(len(y))
            return np.polyfit(x, y, 1)[0] if len(y) > 1 else 0
        return series.rolling(window, min_periods=2).apply(calc_slope, raw=True)
    for col in value_cols:
        for window in windows:
            df[f'{col}_trend_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: rolling_slope(x, window))
    return df

def build_enhanced_features(data, enc_stats=None):
    df = data.copy()
    if enc_stats is not None:
        for c in ['sub_category', 'sub_code']:
            df[c + '_enc'] = df[c].map(enc_stats[c]).fillna(enc_stats['global_mean'])

    df['d_al_am'] = df['feature_al'] - df['feature_am']
    df['r_al_am'] = df['feature_al'] / (df['feature_am'] + 1e-7)
    df['d_cg_by'] = df['feature_cg'] - df['feature_by']
    df['p_am_bz'] = df['feature_am'] * df['feature_bz']
    df['p_am_cd'] = df['feature_am'] * df['feature_cd']
    df['d_j_bz']  = df['feature_j']  - df['feature_bz']
    df['r_l_bq']  = df['feature_l']  / (df['feature_bq'] + 1e-7)

    norm_cols = [
        'feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am',
        'p_am_bz', 'p_am_cd', 'd_j_bz', 'r_l_bq'
    ]
    for col in norm_cols:
        if col in df.columns:
            g = df.groupby('ts_index')[col]
            df[col + '_cs'] = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)

    df['t_cycle'] = np.sin(2 * np.pi * df['ts_index'] / 100)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_trend_features(df)

    for col in ['feature_al', 'feature_am']:
        df[f'{col}_diff_1'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].diff(1)
        df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)

    for col in ['feature_bz', 'feature_cg', 'd_al_am']:
        if col in df.columns:
            df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)

    df = df.fillna(0)
    return df

def get_feature_columns(df):
    exclude_cols = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude_cols]

print('Feature engineering ready')

Feature engineering ready


In [4]:
def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.array(y_target)
    y_pred = np.array(y_pred)
    w = np.array(w)
    denom = np.sum(w * (y_target ** 2))
    if denom <= 0:
        return 0.0
    numerator = np.sum(w * ((y_target - y_pred) ** 2))
    ratio = numerator / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Metric ready')

Metric ready


In [5]:
print('Computing statistics...')
temp = pd.read_parquet(TRAIN_PATH, columns=['code', 'sub_category', 'sub_code', 'weight', 'y_target', 'ts_index'])

code_weight = temp.groupby('code')['weight'].sum()
total_weight = code_weight.sum()
code_weight_pct = (code_weight / total_weight * 100).sort_values(ascending=False)
cumsum = 0
HIGH_WEIGHT_CODES = []
for code, pct in code_weight_pct.items():
    HIGH_WEIGHT_CODES.append(code)
    cumsum += pct
    if cumsum >= 90:
        break
print(f'High-weight codes ({cumsum:.1f}%): {HIGH_WEIGHT_CODES}')

# Use first fold's train data for encoding stats
train_only = temp[temp.ts_index <= FOLDS[0]['train_end']]
train_stats = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': train_only['y_target'].mean()
}
del temp, train_only
gc.collect()
print('Statistics computed')

Computing statistics...
High-weight codes (97.8%): ['83EG83KQ', 'SJZP0OVU', 'MLAAMU3K']
Statistics computed


In [6]:
def train_fold_models(X_fit, y_fit, w_fit, X_hold, y_hold, w_hold, te_features, params, seeds, label):
    """Train seed ensemble for one fold, return val and test predictions."""
    val_pred = np.zeros(len(y_hold))
    tst_pred = np.zeros(len(te_features))
    for seed in seeds:
        mdl = lgb.LGBMRegressor(**params, random_state=seed)
        mdl.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_hold, y_hold)],
            eval_sample_weight=[w_hold],
            callbacks=[lgb.early_stopping(200, verbose=False)]
        )
        val_pred += mdl.predict(X_hold) / len(seeds)
        tst_pred += mdl.predict(te_features) / len(seeds)
        print(f'      {label} seed {seed}: best_iter={mdl.best_iteration_}')
    del mdl
    gc.collect()
    return val_pred, tst_pred


def train_combined_kfold(tr_df, te_df, feature_cols):
    """K-Fold training for combined model (H1, H25)."""
    tst_pred_sum = np.zeros(len(te_df))
    fold_scores = []

    for fi, fold in enumerate(FOLDS):
        print(f'\n  --- Fold {fi+1}/{len(FOLDS)} (train<={fold["train_end"]}, val={fold["val_start"]}-{fold["val_end"]}) ---')
        fit_mask = tr_df.ts_index <= fold['train_end']
        val_mask = (tr_df.ts_index >= fold['val_start']) & (tr_df.ts_index <= fold['val_end'])

        if val_mask.sum() == 0:
            print('  No val samples, skipping')
            continue

        val_pred, tst_pred = train_fold_models(
            tr_df.loc[fit_mask, feature_cols], tr_df.loc[fit_mask, 'y_target'],
            tr_df.loc[fit_mask, 'weight'].values,
            tr_df.loc[val_mask, feature_cols], tr_df.loc[val_mask, 'y_target'],
            tr_df.loc[val_mask, 'weight'].values,
            te_df[feature_cols], RANGE_PARAMS, SEEDS, f'F{fi+1}'
        )
        tst_pred_sum += tst_pred
        fold_score = weighted_rmse_score(tr_df.loc[val_mask, 'y_target'], val_pred, tr_df.loc[val_mask, 'weight'])
        fold_scores.append(fold_score)
        print(f'    Fold {fi+1} score: {fold_score:.5f}')

    tst_pred_avg = tst_pred_sum / len(FOLDS)
    avg_score = np.mean(fold_scores)
    print(f'\n  Average CV score: {avg_score:.5f}')
    return tst_pred_avg, te_df['id'].values, avg_score


def train_two_model_kfold(tr_df, te_df, feature_cols):
    """K-Fold training for two-model approach (H3, H10)."""
    hw_test = te_df['code'].isin(HIGH_WEIGHT_CODES)
    prec_tst_sum = np.zeros(hw_test.sum())
    range_tst_sum = np.zeros((~hw_test).sum())
    fold_scores = []

    for fi, fold in enumerate(FOLDS):
        print(f'\n  --- Fold {fi+1}/{len(FOLDS)} (train<={fold["train_end"]}, val={fold["val_start"]}-{fold["val_end"]}) ---')
        fit_mask = tr_df.ts_index <= fold['train_end']
        val_mask = (tr_df.ts_index >= fold['val_start']) & (tr_df.ts_index <= fold['val_end'])

        if val_mask.sum() == 0:
            print('  No val samples, skipping')
            continue

        hw_fit = fit_mask & tr_df['code'].isin(HIGH_WEIGHT_CODES)
        hw_val = val_mask & tr_df['code'].isin(HIGH_WEIGHT_CODES)
        lw_fit = fit_mask & ~tr_df['code'].isin(HIGH_WEIGHT_CODES)
        lw_val = val_mask & ~tr_df['code'].isin(HIGH_WEIGHT_CODES)

        # Precision model
        print(f'    PRECISION ({hw_fit.sum():,} train, {hw_val.sum():,} val)')
        prec_val, prec_tst = train_fold_models(
            tr_df.loc[hw_fit, feature_cols], tr_df.loc[hw_fit, 'y_target'],
            tr_df.loc[hw_fit, 'weight'].values,
            tr_df.loc[hw_val, feature_cols], tr_df.loc[hw_val, 'y_target'],
            tr_df.loc[hw_val, 'weight'].values,
            te_df.loc[hw_test, feature_cols], PRECISION_PARAMS, SEEDS, f'P{fi+1}'
        )
        prec_tst_sum += prec_tst

        # Range model
        print(f'    RANGE ({lw_fit.sum():,} train, {lw_val.sum():,} val)')
        range_val, range_tst = train_fold_models(
            tr_df.loc[lw_fit, feature_cols], tr_df.loc[lw_fit, 'y_target'],
            tr_df.loc[lw_fit, 'weight'].values,
            tr_df.loc[lw_val, feature_cols], tr_df.loc[lw_val, 'y_target'],
            tr_df.loc[lw_val, 'weight'].values,
            te_df.loc[~hw_test, feature_cols], RANGE_PARAMS, SEEDS, f'R{fi+1}'
        )
        range_tst_sum += range_tst

        # Merge for fold score
        merged_val = np.zeros(val_mask.sum())
        val_idx = tr_df.index[val_mask]
        hw_val_local = tr_df.loc[val_idx, 'code'].isin(HIGH_WEIGHT_CODES).values
        merged_val[hw_val_local] = prec_val
        merged_val[~hw_val_local] = range_val
        fold_score = weighted_rmse_score(tr_df.loc[val_mask, 'y_target'], merged_val, tr_df.loc[val_mask, 'weight'])
        fold_scores.append(fold_score)
        print(f'    Fold {fi+1} merged score: {fold_score:.5f}')

    # Average test predictions across folds
    merged_tst = np.zeros(len(te_df))
    merged_tst[hw_test.values] = prec_tst_sum / len(FOLDS)
    merged_tst[~hw_test.values] = range_tst_sum / len(FOLDS)

    avg_score = np.mean(fold_scores)
    print(f'\n  Average CV score: {avg_score:.5f}')
    return merged_tst, te_df['id'].values, avg_score


def train_single_horizon(horizon):
    print(f'\n{"="*60}')
    print(f'Training Horizon {horizon}')
    print(f'{"="*60}')

    tr_df = pd.read_parquet(TRAIN_PATH).query(f'horizon == {horizon}')
    te_df = pd.read_parquet(TEST_PATH).query(f'horizon == {horizon}')
    tr_df = build_enhanced_features(tr_df, train_stats)
    te_df = build_enhanced_features(te_df, train_stats)
    feature_cols = get_feature_columns(tr_df)
    print(f'  Features: {len(feature_cols)}')

    if horizon in TWO_MODEL_HORIZONS:
        print(f'  Strategy: TWO-MODEL K-FOLD')
        tst_pred, test_ids, score = train_two_model_kfold(tr_df, te_df, feature_cols)
    else:
        print(f'  Strategy: COMBINED K-FOLD')
        tst_pred, test_ids, score = train_combined_kfold(tr_df, te_df, feature_cols)

    print(f'\n  Horizon {horizon} FINAL CV score: {score:.5f}')

    del tr_df, te_df
    gc.collect()
    return tst_pred, test_ids, score

print('Training function ready — K-Fold')

Training function ready — K-Fold


In [7]:
test_outputs = []
validation_scores = {}

for hz in FORECAST_WINDOWS:
    tst_pred, test_ids, val_score = train_single_horizon(hz)
    test_outputs.append(pd.DataFrame({'id': test_ids, 'prediction': tst_pred}))
    validation_scores[hz] = val_score

print('\nAll horizons complete')


Training Horizon 1
  Features: 148
  Strategy: COMBINED K-FOLD

  --- Fold 1/3 (train<=2800, val=2801-3100) ---
      F1 seed 42: best_iter=207
      F1 seed 2024: best_iter=179
      F1 seed 12345: best_iter=246
    Fold 1 score: 0.05880

  --- Fold 2/3 (train<=3100, val=3101-3350) ---
      F2 seed 42: best_iter=82
      F2 seed 2024: best_iter=220
      F2 seed 12345: best_iter=170
    Fold 2 score: 0.04485

  --- Fold 3/3 (train<=3350, val=3351-3601) ---
      F3 seed 42: best_iter=152
      F3 seed 2024: best_iter=110
      F3 seed 12345: best_iter=123
    Fold 3 score: 0.05631

  Average CV score: 0.05332

  Horizon 1 FINAL CV score: 0.05332

Training Horizon 3
  Features: 148
  Strategy: TWO-MODEL K-FOLD

  --- Fold 1/3 (train<=2800, val=2801-3100) ---
    PRECISION (111,520 train, 11,302 val)
      P1 seed 42: best_iter=154
      P1 seed 2024: best_iter=225
      P1 seed 12345: best_iter=107
    RANGE (921,134 train, 120,478 val)
      R1 seed 42: best_iter=200
      R1 seed 2

In [8]:
submission = pd.concat(test_outputs, ignore_index=True)
submission.to_csv('submission.csv', index=False)

print(f'\n{"="*55}')
print(f'{"Horizon":<10} {"Strategy":<18} {"Avg CV Score":<12}')
print('-'*42)
for hz in sorted(validation_scores.keys()):
    strategy = 'TwoModel-KFold' if hz in TWO_MODEL_HORIZONS else 'Combined-KFold'
    print(f'{hz:<10} {strategy:<18} {validation_scores[hz]:<12.5f}')
print('-'*42)
avg_val = np.mean(list(validation_scores.values()))
print(f'{"Avg":<10} {"":<18} {avg_val:<12.5f}')
print(f'{"="*55}')
print(f'\nSaved {len(submission):,} predictions to submission.csv')
print(f'Prev best: 0.2633 (test)')


Horizon    Strategy           Avg CV Score
------------------------------------------
1          Combined-KFold     0.05332     
3          TwoModel-KFold     0.09009     
10         TwoModel-KFold     0.14395     
25         Combined-KFold     0.19195     
------------------------------------------
Avg                           0.11983     

Saved 1,447,107 predictions to submission.csv
Prev best: 0.2633 (test)
